In [7]:
import fastf1
from pathlib import Path

CACHE_DIR = Path("../../data_raw/f1_cache")
fastf1.Cache.enable_cache(CACHE_DIR)

print("FastF1 cache:", fastf1.Cache.get_cache_dir())

AttributeError: type object 'Cache' has no attribute 'get_cache_dir'

In [1]:
# === Cell 1: Config ===

import os, json, datetime
import numpy as np
import pandas as pd
import fastf1 as ff1

from pathlib import Path
import datetime


# Paths (align to your repo structure)
CACHE_DIR = "../../data_raw/f1_cache"
OUT_DIR   = "../../data_processed"

os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

# Enable cache (shared across notebooks)
ff1.Cache.enable_cache(CACHE_DIR)

YEARS = list(range(2018, 2026))
PRACTICE = ["FP1", "FP2", "FP3"]
QUALI = "Q"

# Output files
OUT_A = os.path.join(OUT_DIR, "SG_quali_2025.parquet")

META_A = OUT_A.replace(".parquet", ".meta.json")



In [2]:
#i just want to check paths, trauma from when the cache went all over the place, wrong folder, went to tmp fml


CACHE_DIR = Path(CACHE_DIR)   # <-- this is the missing line

test_file = CACHE_DIR / "_cache_path_test.txt"
with open(test_file, "w") as f:
    f.write(f"Cache test written at {datetime.datetime.now().isoformat()}\n")

print("Test file written to:", test_file)


OUT_DIR = Path(OUT_DIR)   # <-- this is the missing line
testp_file = OUT_DIR / "_parquets_path_test.txt"
with open(testp_file, "w") as f:
    f.write(f"Parquets test written at {datetime.datetime.now().isoformat()}\n")

print("Test file written to:", testp_file)

Test file written to: ..\..\data_raw\f1_cache\_cache_path_test.txt
Test file written to: ..\..\data_processed\_parquets_path_test.txt


In [ ]:
#trial test 2025 round 1 laps TRUE and weather TRUE and Tele TRUE

year = 2025
round_no = 18

for session_code in ["Q"]:
    print(f"Loading {year} R{round_no} {session_code}...")
    ses = ff1.get_session(year, round_no, session_code)
    ses.load(
        laps=True,
        weather= True,
        messages=False,
        telemetry=True
    )
    print(
        f"  laps: {len(ses.laps)} | "
        f"drivers: {ses.laps['DriverNumber'].nunique()}")

core           INFO 	Loading data for Singapore Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


Loading 2025 R18 Q...


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	No cached data found for car_data. Loading data...
_api           INFO 	Fetching car data...
_api           INFO 	Parsing car data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	Fetching position data...
_api           INFO 	Parsing position data...
req            INFO 	Data has been written to cache!
req            INFO 	Using cached data for weather_data
core           INFO 	Finished loading data for 20 drivers: ['63', '1', '81', '12', '4', '44', '16', '6', '87', '14', '27', '30', '22', '5', '18', '43', '31', '10', '23', '55']


  laps: 298 | drivers: 20


In [11]:
from pathlib import Path

sg_quali_2025 = Path("../../data_raw/f1_cache") / "2025" / "2025-10-05_Singapore_Grand_Prix" / "2025-10-04_Qualifying"

print("Exists:", sg_quali_2025.exists())
print("Files:")
for f in sg_quali_2025.glob("*"):
    print(" -", f.name)

Exists: True
Files:
 - car_data.ff1pkl
 - driver_info.ff1pkl
 - position_data.ff1pkl
 - session_info.ff1pkl
 - session_status_data.ff1pkl
 - timing_app_data.ff1pkl
 - track_status_data.ff1pkl
 - weather_data.ff1pkl
 - _extended_timing_data.ff1pkl


In [12]:

# Write into a structured subfolder under OUT_DIR
out_dir = OUT_DIR / "singapore_gp" / "quali" / f"year={year}"
out_dir.mkdir(parents=True, exist_ok=True)

# ----------------------------
# 1) LAPS parquet
# ----------------------------
laps = ses.laps.copy()

keep_cols = [
    "Driver", "DriverNumber", "Team",
    "LapNumber", "LapTime",
    "Compound", "TyreLife", "FreshTyre",
    "IsAccurate",
    "Sector1Time", "Sector2Time", "Sector3Time",
    "TrackStatus"
]
laps_out = laps[[c for c in keep_cols if c in laps.columns]].copy()

laps_out["Year"] = year
laps_out["RoundNumber"] = round_no
laps_out["Session"] = session_code

# Stable join key
laps_out["LapId"] = (
    laps_out["Year"].astype(str) + "_" +
    laps_out["RoundNumber"].astype(str) + "_" +
    laps_out["Session"].astype(str) + "_" +
    laps_out["DriverNumber"].astype(str) + "_" +
    laps_out["LapNumber"].astype(str)
)

laps_out.to_parquet(out_dir / "laps.parquet", index=False)
print("Saved:", (out_dir / "laps.parquet").resolve(), "| rows:", len(laps_out))

# ----------------------------
# 2) TELEMETRY parquet
# ----------------------------
TEL_COLS = ["Time", "Distance", "Speed", "Throttle", "Brake", "SteeringAngle", "X", "Y"]

telemetry_rows = []
bad = 0

# Good default for Quali-like analysis; swap to `laps` if you want all laps
laps_for_tel = laps.pick_quicklaps()

for _, lap in laps_for_tel.iterlaps():
    try:
        tel = lap.get_telemetry()[[c for c in TEL_COLS if c in lap.get_telemetry().columns]].dropna()

        lap_id = f"{year}_{round_no}_{session_code}_{lap['DriverNumber']}_{lap['LapNumber']}"

        tel = tel.copy()
        # parquet-friendly numeric time
        if "Time" in tel.columns:
            tel["TimeSeconds"] = tel["Time"].dt.total_seconds()
            tel.drop(columns=["Time"], inplace=True)

        tel["LapId"] = lap_id
        tel["Driver"] = lap["Driver"]
        tel["DriverNumber"] = lap["DriverNumber"]
        tel["LapNumber"] = lap["LapNumber"]

        telemetry_rows.append(tel)

    except Exception:
        bad += 1

telemetry_out = pd.concat(telemetry_rows, ignore_index=True)
telemetry_out.to_parquet(out_dir / "telemetry.parquet", index=False)

print("Saved:", (out_dir / "telemetry.parquet").resolve(),
      "| rows:", len(telemetry_out),
      "| bad laps skipped:", bad)

# ----------------------------
# 3) Quick sanity check (optional)
# ----------------------------
print("Unique LapId in telemetry:", telemetry_out["LapId"].nunique())

Saved: C:\Users\aadha\OneDrive - Singapore Management University\Y2\S2\New folder\DAP\DAP F1 work\Working directory\data_processed\singapore_gp\quali\year=2025\laps.parquet | rows: 298
Saved: C:\Users\aadha\OneDrive - Singapore Management University\Y2\S2\New folder\DAP\DAP F1 work\Working directory\data_processed\singapore_gp\quali\year=2025\telemetry.parquet | rows: 61255 | bad laps skipped: 0
Unique LapId in telemetry: 89
